# Experimentos - MLP do Zero

Este notebook registra os experimentos do projeto. A implementacao principal esta no pacote `mlp/` e os treinos podem ser executados pelos scripts em `scripts/`.

## Configuracoes comparadas

| Experimento | Arquitetura | Ativacao | Learning rate | Momentum |
| --- | --- | --- | --- | --- |
| baseline_relu | 784-256-128-10 | ReLU | 0.05 | 0.9 |
| deeper_relu | 784-256-128-64-10 | ReLU | 0.03 | 0.9 |

## Diagnostico de overfitting

A coluna `train_acc` pode chegar a `1.0000` porque a rede viu esses exemplos durante o treinamento. Para avaliar generalizacao, compare `val_acc` e `test_acc`. Se `train_acc` for muito maior que `val_acc`/`test_acc`, ha sinal de overfitting.

Nos resultados atuais, as duas redes chegaram a 100% no treino, mas ficaram perto de 98% em validacao/teste. Isso indica overfitting leve, nao erro na metrica nem vazamento obvio de dados.


In [ ]:
# Rode esta celula depois de instalar as dependencias.
# Ela executa os dois experimentos definidos em scripts/run_experiments.py.

import subprocess
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

subprocess.run([sys.executable, '-m', 'scripts.run_experiments'], cwd=project_root, check=True)


In [ ]:
import csv
import json
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

def read_csv(path):
    with path.open(encoding='utf-8') as file:
        return list(csv.DictReader(file))

def print_table(rows, columns):
    if not rows:
        print('Nenhum resultado encontrado. Rode a celula de experimentos primeiro.')
        return
    widths = {col: max(len(col), *(len(str(row.get(col, ''))) for row in rows)) for col in columns}
    print(' | '.join(col.ljust(widths[col]) for col in columns))
    print('-+-'.join('-' * widths[col] for col in columns))
    for row in rows:
        print(' | '.join(str(row.get(col, '')).ljust(widths[col]) for col in columns))

def fmt(value):
    if value is None or value == '':
        return ''
    if isinstance(value, int):
        return str(value)
    return f'{float(value):.4f}'

def diagnostics_from_metrics(metrics):
    history = metrics['history']
    test = metrics['test']
    diagnostics = metrics.get('diagnostics', {}).copy()
    diagnostics.setdefault('final_train_accuracy', history['accuracy'][-1])
    diagnostics.setdefault('test_accuracy', test['accuracy'])
    if history.get('val_accuracy'):
        best_idx = max(range(len(history['val_accuracy'])), key=lambda idx: history['val_accuracy'][idx])
        diagnostics.setdefault('final_val_accuracy', history['val_accuracy'][-1])
        diagnostics.setdefault('best_val_accuracy', history['val_accuracy'][best_idx])
        diagnostics.setdefault('best_val_epoch', best_idx + 1)
        diagnostics.setdefault('train_val_accuracy_gap', diagnostics['final_train_accuracy'] - diagnostics['final_val_accuracy'])
        diagnostics.setdefault('train_test_accuracy_gap', diagnostics['final_train_accuracy'] - diagnostics['test_accuracy'])
    return diagnostics

run_names = ['baseline_relu', 'deeper_relu']
diagnostic_rows = []
metric_rows = []
for run_name in run_names:
    metrics_path = project_root / 'results' / f'{run_name}_metrics.json'
    if not metrics_path.exists():
        continue
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    config = metrics['config']
    test = metrics['test']
    diagnostics = diagnostics_from_metrics(metrics)
    diagnostic_rows.append({
        'run_name': run_name,
        'layers': config['layers'],
        'train_acc': fmt(diagnostics.get('final_train_accuracy')),
        'val_acc': fmt(diagnostics.get('final_val_accuracy')),
        'test_acc': fmt(diagnostics.get('test_accuracy')),
        'gap_train_val': fmt(diagnostics.get('train_val_accuracy_gap')),
        'gap_train_test': fmt(diagnostics.get('train_test_accuracy_gap')),
        'best_val_acc': fmt(diagnostics.get('best_val_accuracy')),
        'best_val_epoch': diagnostics.get('best_val_epoch', ''),
    })
    metric_rows.append({
        'run_name': run_name,
        'accuracy': fmt(test.get('accuracy')),
        'loss': fmt(test.get('loss')),
        'precision_macro': fmt(test.get('precision_macro')),
        'recall_macro': fmt(test.get('recall_macro')),
        'f1_macro': fmt(test.get('f1_macro')),
        'balanced_accuracy': fmt(test.get('balanced_accuracy')),
    })

print('Diagnostico de overfitting')
print_table(diagnostic_rows, ['run_name', 'layers', 'train_acc', 'val_acc', 'test_acc', 'gap_train_val', 'gap_train_test', 'best_val_acc', 'best_val_epoch'])
print('\nMetricas no conjunto de teste')
print_table(metric_rows, ['run_name', 'accuracy', 'loss', 'precision_macro', 'recall_macro', 'f1_macro', 'balanced_accuracy'])


## Relatorios por digito

As tabelas abaixo mostram precision, recall, F1 e support para cada digito. Recall e especialmente util aqui porque mostra, para cada classe real, quantos exemplos daquele digito a rede conseguiu recuperar corretamente.

In [ ]:
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

for run_name in ['baseline_relu', 'deeper_relu']:
    report_path = project_root / 'results' / f'{run_name}_classification_report.csv'
    print(f'\n{run_name}')
    if report_path.exists():
        rows = read_csv(report_path)
        print_table(rows, ['class', 'precision', 'recall', 'f1', 'support'])
    else:
        print('Relatorio ainda nao encontrado. Rode os experimentos primeiro.')


## Curvas

Depois do treino, abra as imagens em `results/` para analisar se a loss convergiu e se a acuracia de validacao acompanhou a de treino sem overfitting forte.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

for run_name in ['baseline_relu', 'deeper_relu']:
    print('\n' + run_name + ' - curvas')
    curves_path = project_root / 'results' / f'{run_name}_curves.png'
    if curves_path.exists():
        display(Image(filename=str(curves_path)))
    print(run_name + ' - matriz de confusao')
    confusion_path = project_root / 'results' / f'{run_name}_confusion.png'
    if confusion_path.exists():
        display(Image(filename=str(confusion_path)))


## Analise

A configuracao com melhor acuracia de teste foi `deeper_relu`, com `0.9823`. A `baseline_relu` ficou muito proxima, com `0.9812`, entao a camada extra trouxe ganho pequeno.

A loss caiu de forma consistente nas duas configuracoes. No final das 15 epocas, ambas chegaram a `1.0000` de acuracia no treino, mas validacao e teste ficaram em torno de `0.981` a `0.982`. Isso mostra overfitting leve: a rede memorizou o conjunto de treino, mas ainda generalizou bem.

A meta de `92%` foi atingida com folga. A matriz de confusao confirma que o teste nao ficou perfeito: a `baseline_relu` errou 188 imagens de teste e a `deeper_relu` errou 177.

Para uma entrega mais conservadora, eu consideraria usar early stopping perto da melhor epoca de validacao ou reduzir o numero de epocas. Mantive 15 epocas porque o requisito principal era ultrapassar 92% e comparar duas configuracoes completas.
